# 05a - Metode Ansambel: Bagging dan Random Forest
Materi Modul 5: metode berbasis ansambel dan bagging, random forest, lab.

- **Bagging** (Bootstrap Aggregating): latih banyak model pada sampel bootstrap, lalu gabungkan prediksinya (voting). Menurunkan **varians**.
- **Random Forest**: bagging pohon + pemilihan **fitur acak** di tiap split -> pohon lebih beragam.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier

df = pd.read_csv('../datasets/breast_cancer.csv')
X, y = df.drop(columns='target'), df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

## 1. Satu pohon vs bagging

In [ ]:
pohon = DecisionTreeClassifier(random_state=42)
bagging = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=42)

print('Satu pohon | CV =', round(cross_val_score(pohon, X_train, y_train, cv=5).mean(), 4))
print('Bagging    | CV =', round(cross_val_score(bagging, X_train, y_train, cv=5).mean(), 4))

bagging.fit(X_train, y_train)
print('Bagging OOB score:', round(bagging.oob_score_, 4), '(estimasi gratis dari data yang tidak terpilih bootstrap)')

## 2. Pengaruh jumlah pohon

In [ ]:
jumlah = [1, 5, 10, 25, 50, 100, 200]
acc = []
for n in jumlah:
    rf = RandomForestClassifier(n_estimators=n, random_state=42).fit(X_train, y_train)
    acc.append(rf.score(X_test, y_test))

plt.plot(jumlah, acc, marker='o')
plt.xscale('log'); plt.xlabel('n_estimators'); plt.ylabel('Akurasi test')
plt.title('Random Forest: lebih banyak pohon'); plt.show()

## 3. Random Forest dan OOB

In [ ]:
rf = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=42).fit(X_train, y_train)
print('OOB score    :', round(rf.oob_score_, 4))
print('Akurasi test :', round(rf.score(X_test, y_test), 4))

## 4. Pengaruh `max_features` dan `max_depth`

In [ ]:
for mf in ['sqrt', 'log2', 0.5, 1.0]:
    rf = RandomForestClassifier(n_estimators=100, max_features=mf, random_state=42)
    print(f'max_features={str(mf):5s} | CV = {cross_val_score(rf, X_train, y_train, cv=5).mean():.4f}')
print()
for md in [2, 4, 8, None]:
    rf = RandomForestClassifier(n_estimators=100, max_depth=md, random_state=42)
    print(f'max_depth={str(md):5s} | CV = {cross_val_score(rf, X_train, y_train, cv=5).mean():.4f}')

## 5. Feature importance

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
imp.tail(10).plot(kind='barh', figsize=(7, 4))
plt.title('10 fitur terpenting (Random Forest)'); plt.show()

## Catatan
- (kenapa bagging menurunkan varians, apa bedanya RF dengan bagging biasa, apa itu OOB, dll.)